# Fase 4: Modelo Dimensional (Esquema en Estrella) para el Tablero de BI

Construye en PostgreSQL un **modelo en estrella** a partir de la tabla `despachos_enriquecidos` cargada en `Fase1_ETL`. Es la base del tablero de Power BI (objetivo 2).

```mermaid
erDiagram
    fact_despachos }o--|| dim_calendario : fecha_key
    fact_despachos }o--|| dim_origen : origen_key
    fact_despachos }o--|| dim_destino : destino_key
    fact_despachos }o--|| dim_flota : flota_key
    fact_despachos {
        bigint despacho_id PK
        int fecha_key FK
        string origen_key FK
        string destino_key FK
        int flota_key FK
        float pasajeros
        float capacidad_total
        float pax_km
        float emisiones_kg_co2e
    }
    dim_calendario {
        int fecha_key PK
        date fecha
        string dia_semana
        bool es_festivo
    }
    dim_origen {
        string geo_key PK
        string municipio
        string departamento
    }
    dim_destino {
        string geo_key PK
        string municipio
        string departamento
    }
    dim_flota {
        int flota_key PK
        string clase_vehiculo
        string nivel_servicio
        int capacidad_bus
    }
```

| Tabla | Grano / contenido | Llave |
|---|---|---|
| `fact_despachos` | Un registro de despacho (terminal, fecha, hora, ruta) | `despacho_id` |
| `dim_calendario` | Un día del periodo, con atributos de calendario y festivos | `fecha_key` (AAAAMMDD) |
| `dim_geografia` | Un municipio (código DANE) con su departamento | `geo_key` |
| `dim_origen`, `dim_destino` | Vistas de `dim_geografia` (dimensión de rol) para origen y destino | `geo_key` |
| `dim_flota` | Combinación de clase de vehículo y nivel de servicio | `flota_key` |

Las medidas de emisiones se calculan con las mismas constantes de `Fase3_Modelado_Ambiental` (10 km/galón, 10,15 kg CO2e/galón, ocupación ideal del 80 %).

In [1]:
import os
import pandas as pd
import holidays
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from IPython.display import display

# Credenciales de PostgreSQL desde ../.env (mismo esquema que Fase1_ETL; ver .env.example)
load_dotenv("../.env")
PG = dict(user=os.getenv("POSTGRES_USER", "postgres"), pwd=os.environ["POSTGRES_PASSWORD"],
          host=os.getenv("POSTGRES_HOST", "localhost"), port=os.getenv("POSTGRES_PORT", "5433"),
          db=os.getenv("POSTGRES_DB", "movilidad"))
engine = create_engine(f"postgresql+psycopg2://{PG['user']}:{PG['pwd']}@{PG['host']}:{PG['port']}/{PG['db']}")
with engine.connect() as conn:
    print("Conexión a PostgreSQL OK")

TABS = Path("../results/tables"); TABS.mkdir(parents=True, exist_ok=True)
RENDIMIENTO_KM_GAL = 10.0
FACTOR_EMISION = 10.15
OCUPACION_IDEAL = 0.8

def ejecutar(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

Conexión a PostgreSQL OK


## 1. Dimensión calendario

In [2]:
with engine.connect() as conn:
    rango = conn.execute(text("SELECT MIN(fecha_despacho)::date, MAX(fecha_despacho)::date FROM despachos_enriquecidos")).one()
fechas = pd.date_range(rango[0], rango[1], freq="D")
festivos = holidays.Colombia(years=sorted(set(fechas.year)))
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Septiembre", "Octubre", "Noviembre", "Diciembre"]

cal = pd.DataFrame({"fecha": fechas.date})
cal["fecha_key"] = fechas.strftime("%Y%m%d").astype(int)
cal["anio"] = fechas.year
cal["trimestre"] = fechas.quarter
cal["mes"] = fechas.month
cal["nombre_mes"] = [meses[m - 1] for m in fechas.month]
cal["anio_mes"] = fechas.strftime("%Y-%m")
cal["dia"] = fechas.day
cal["dia_semana_num"] = fechas.dayofweek + 1
cal["dia_semana"] = [dias[d] for d in fechas.dayofweek]
cal["es_fin_semana"] = fechas.dayofweek >= 5
cal["es_festivo"] = [f in festivos for f in fechas.date]
cal["nombre_festivo"] = [festivos.get(f, "") for f in fechas.date]
cal["mes_incompleto"] = cal["anio_mes"] == "2025-12"   # datos parciales en la fuente
cal = cal[["fecha_key", "fecha", "anio", "trimestre", "mes", "nombre_mes", "anio_mes", "dia",
           "dia_semana_num", "dia_semana", "es_fin_semana", "es_festivo", "nombre_festivo", "mes_incompleto"]]

ejecutar("DROP VIEW IF EXISTS dim_origen, dim_destino; DROP TABLE IF EXISTS fact_despachos, dim_calendario, dim_geografia, dim_flota CASCADE;")
cal.to_sql("dim_calendario", engine, index=False, if_exists="replace")
ejecutar("ALTER TABLE dim_calendario ADD PRIMARY KEY (fecha_key);")
print(f"dim_calendario: {len(cal)} días, {cal.es_festivo.sum()} festivos.")

dim_calendario: 455 días, 20 festivos.


## 2. Dimensiones de geografía y flota

In [3]:
ejecutar("""
CREATE TABLE dim_geografia AS
SELECT codigo AS geo_key, MAX(municipio) AS municipio, MAX(codigo_dept) AS codigo_dept, MAX(departamento) AS departamento
FROM (
    SELECT codigo_origen AS codigo, nombre_origen AS municipio, codigo_dept_origen AS codigo_dept, nombre_dept_origen AS departamento
    FROM despachos_enriquecidos
    UNION
    SELECT codigo_destino, nombre_destino, codigo_dept_destino, nombre_dept_destino
    FROM despachos_enriquecidos
) g
GROUP BY codigo;
ALTER TABLE dim_geografia ADD PRIMARY KEY (geo_key);

CREATE VIEW dim_origen AS
SELECT geo_key, municipio AS municipio_origen, codigo_dept AS codigo_dept_origen, departamento AS departamento_origen FROM dim_geografia;
CREATE VIEW dim_destino AS
SELECT geo_key, municipio AS municipio_destino, codigo_dept AS codigo_dept_destino, departamento AS departamento_destino FROM dim_geografia;

CREATE TABLE dim_flota AS
SELECT ROW_NUMBER() OVER (ORDER BY clase_vehiculo, nivel_servicio)::int AS flota_key,
       clase_vehiculo, nivel_servicio, MAX(capacidad_estimada)::int AS capacidad_bus
FROM despachos_enriquecidos
GROUP BY clase_vehiculo, nivel_servicio;
ALTER TABLE dim_flota ADD PRIMARY KEY (flota_key);
""")
display(pd.read_sql("SELECT COUNT(*) AS municipios FROM dim_geografia", engine))
display(pd.read_sql("SELECT * FROM dim_flota", engine))

,municipios
0,674


,flota_key,clase_vehiculo,nivel_servicio,capacidad_bus
0,1,BUS,BASICO,40
1,2,BUS,LUJO,40
2,3,BUS,PREFERENCIAL DE LUJO,40


## 3. Tabla de hechos

In [4]:
ejecutar(f"""
CREATE TABLE fact_despachos AS
SELECT ROW_NUMBER() OVER (ORDER BY e.fecha_despacho, e.terminal, e.hora_despacho)::bigint AS despacho_id,
       TO_CHAR(e.fecha_despacho, 'YYYYMMDD')::int        AS fecha_key,
       e.hora_despacho,
       e.codigo_origen                                   AS origen_key,
       e.codigo_destino                                  AS destino_key,
       f.flota_key,
       e.terminal,
       e.tipo_despacho,
       e.despachos,
       e.pasajeros,
       e.capacidad_total,
       e.distancia_km,
       e.pax_km,
       e.distancia_km / {RENDIMIENTO_KM_GAL} * e.despachos                     AS consumo_galones,
       e.distancia_km / {RENDIMIENTO_KM_GAL} * e.despachos * {FACTOR_EMISION}  AS emisiones_kg_co2e,
       e.capacidad_total * {OCUPACION_IDEAL} * e.distancia_km                  AS pax_km_ideal,
       e.factor_ocupacion_pct,
       e.viaje_vacio,
       e.sobrecupo
FROM despachos_enriquecidos e
JOIN dim_flota f
  ON f.clase_vehiculo IS NOT DISTINCT FROM e.clase_vehiculo
 AND f.nivel_servicio IS NOT DISTINCT FROM e.nivel_servicio;

ALTER TABLE fact_despachos ADD PRIMARY KEY (despacho_id);
ALTER TABLE fact_despachos ADD CONSTRAINT fk_fecha   FOREIGN KEY (fecha_key)   REFERENCES dim_calendario (fecha_key);
ALTER TABLE fact_despachos ADD CONSTRAINT fk_origen  FOREIGN KEY (origen_key)  REFERENCES dim_geografia (geo_key);
ALTER TABLE fact_despachos ADD CONSTRAINT fk_destino FOREIGN KEY (destino_key) REFERENCES dim_geografia (geo_key);
ALTER TABLE fact_despachos ADD CONSTRAINT fk_flota   FOREIGN KEY (flota_key)   REFERENCES dim_flota (flota_key);
CREATE INDEX ix_fact_fecha ON fact_despachos (fecha_key);
CREATE INDEX ix_fact_ruta  ON fact_despachos (origen_key, destino_key);
""")
display(pd.read_sql("SELECT COUNT(*) AS registros FROM fact_despachos", engine))

,registros
0,5333855


## 4. Validación de totales: fuente frente a modelo en estrella

Para la tabla de hechos y cada KPI principal se comparan el número de registros y los totales de control entre la tabla fuente (`despachos_enriquecidos`), el modelo en estrella y el dataset de la Fase 3. La diferencia esperada es cero. La columna `power_bi` se completa a mano con los valores que muestra el tablero (tarjetas con las mismas medidas).

In [5]:
sql_totales = """
SELECT COUNT(*)::float8 AS registros, SUM(pasajeros)::float8 AS pasajeros, SUM(despachos)::float8 AS buses,
       SUM(pax_km)::float8 AS pax_km, SUM(capacidad_total)::float8 AS capacidad
FROM {tabla}
"""
fuente = pd.read_sql(sql_totales.format(tabla="despachos_enriquecidos"), engine).iloc[0]
estrella = pd.read_sql(sql_totales.format(tabla="fact_despachos"), engine).iloc[0]
emis_estrella = pd.read_sql("SELECT SUM(emisiones_kg_co2e)::float8 AS e FROM fact_despachos", engine).iloc[0, 0]
f3 = pd.read_parquet("../datos/fase3_dataset_emisiones.parquet", columns=["emisiones_brutas_co2e", "pasajeros", "pax_km_real"])

validacion = pd.DataFrame({
    "control": ["Número de registros", "Suma de pasajeros", "Suma de buses despachados", "Suma de pasajeros-km",
                "Ocupación agregada (%)", "Emisiones totales (t CO2e)", "Intensidad (kg CO2e/pax-km)"],
    "fuente": [fuente.registros, fuente.pasajeros, fuente.buses, fuente.pax_km,
               fuente.pasajeros / fuente.capacidad * 100, f3.emisiones_brutas_co2e.sum() / 1000,
               f3.emisiones_brutas_co2e.sum() / f3.pax_km_real.sum()],
    "modelo_estrella": [estrella.registros, estrella.pasajeros, estrella.buses, estrella.pax_km,
                        estrella.pasajeros / estrella.capacidad * 100, emis_estrella / 1000, emis_estrella / estrella.pax_km],
})
validacion["diferencia"] = (validacion["fuente"] - validacion["modelo_estrella"]).round(4)
validacion["power_bi"] = None   # completar con los valores del tablero
validacion.to_csv(TABS / "fase4_validacion_modelo_estrella.csv", index=False)
pd.set_option("display.float_format", "{:,.4f}".format)
display(validacion)

huerfanos = pd.read_sql("""
SELECT
  (SELECT COUNT(*) FROM fact_despachos f LEFT JOIN dim_calendario c USING (fecha_key) WHERE c.fecha_key IS NULL) AS sin_fecha,
  (SELECT COUNT(*) FROM fact_despachos f LEFT JOIN dim_geografia g ON g.geo_key = f.origen_key WHERE g.geo_key IS NULL) AS sin_origen,
  (SELECT COUNT(*) FROM fact_despachos f LEFT JOIN dim_geografia g ON g.geo_key = f.destino_key WHERE g.geo_key IS NULL) AS sin_destino
""", engine)
display(huerfanos)
print("Integridad referencial OK" if huerfanos.sum(axis=1).iloc[0] == 0 else "Hay registros sin dimensión: revisar")

,control,fuente,modelo_estrella,diferencia,power_bi
0,Número de registros,"5,333,855.0000","5,333,855.0000",0.0000,None
1,Suma de pasajeros,"68,978,580.0000","68,978,580.0000",0.0000,None
2,Suma de buses despachados,"5,497,543.0000","5,497,543.0000",0.0000,None
3,Suma de pasajeros-km,"23,281,914,793.4589","23,281,914,793.4583",0.0006,None
4,Ocupación agregada (%),31.3679,31.3679,0.0000,None
5,Emisiones totales (t CO2e),"1,580,459.1365","1,580,459.1365",0.0000,None
6,Intensidad (kg CO2e/pax-km),0.0679,0.0679,-0.0000,None


,sin_fecha,sin_origen,sin_destino
0,0,0,0


Integridad referencial OK


## 5. Medidas DAX para Power BI

Crear estas medidas en la tabla `fact_despachos` (Inicio → Nueva medida). Son las mismas fórmulas del protocolo de evaluación de la Fase 3.

| Medida | DAX |
|---|---|
| Pasajeros | `Pasajeros = SUM(fact_despachos[pasajeros])` |
| Buses despachados | `Buses = SUM(fact_despachos[despachos])` |
| Ocupación % | `Ocupación % = DIVIDE(SUM(fact_despachos[pasajeros]), SUM(fact_despachos[capacidad_total])) * 100` |
| % Registros vacíos | `% Vacíos = DIVIDE(CALCULATE(COUNTROWS(fact_despachos), fact_despachos[viaje_vacio] = TRUE()), COUNTROWS(fact_despachos)) * 100` |
| % Ocupación crítica | `% Crítica = DIVIDE(CALCULATE(COUNTROWS(fact_despachos), fact_despachos[factor_ocupacion_pct] < 35), COUNTROWS(fact_despachos)) * 100` |
| Emisiones (t CO2e) | `Emisiones t = SUM(fact_despachos[emisiones_kg_co2e]) / 1000` |
| Intensidad real | `Intensidad real = DIVIDE(SUM(fact_despachos[emisiones_kg_co2e]), SUM(fact_despachos[pax_km]))` |
| Intensidad ideal | `Intensidad ideal = DIVIDE(SUM(fact_despachos[emisiones_kg_co2e]), SUM(fact_despachos[pax_km_ideal]))` |
| Exceso % | `Exceso % = DIVIDE([Intensidad real] - [Intensidad ideal], [Intensidad ideal]) * 100` |
| Emisiones evitables (t) | `Evitables t = (SUM(fact_despachos[emisiones_kg_co2e]) - SUM(fact_despachos[pax_km]) * [Intensidad ideal]) / 1000` |